# Robustness of Accessibility
## Notebook 2b/4 - (Optional) visualize graph construction

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
import osmnx as ox
import folium

#### Addd rivers for visualization of the flooded areas

In [ ]:
# use name to get bounds
from pathlib import Path
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_outputs_path,
)

cache_dir: Path = get_roa_cache_path()
output_dir: Path = get_roa_outputs_path()
place_name = RoaNotebookConfig.place_name
place_gdf = ox.geocode_to_gdf(place_name)

In [ ]:
# 1. From the GeoDataFrame
if len(place_gdf) > 1:
    raise RuntimeError(f"gdf for {place_name} has multiple entries, could not safely deterine bounds")
boundary_geom = place_gdf.geometry.iloc[0]  # usually the first geometry

# 2. If it’s a MultiPolygon, merge the parts into one

if boundary_geom.geom_type == "MultiPolygon":
    boundary_geom = unary_union(boundary_geom)
elif boundary_geom.geom_type != "Polygon":
    raise RuntimeError(f"Could not combine boundary due to unexpected data structure {boundary_geom.geom_type}")
print(boundary_geom.bounds)
boundary_geom

In [ ]:
tags_rivers = {"water": ["river"]}
rivers = ox.features_from_polygon(polygon=boundary_geom, tags=tags_rivers)

tags_rivers_line = {"waterway": ["river"]}
waterways = ox.features_from_polygon(polygon=boundary_geom, tags=tags_rivers_line)

In [ ]:
# load flooded area:
from pathlib import Path

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    get_roa_hazard_data_path,
)

event = RoaNotebookConfig.event
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

In [ ]:
# Import shapes of flooded areas
hazard_area = gpd.read_file(hazard_data_path)
hazrd_area_geoms = hazard_area.geometry

# Combine all shapes of the flooded are into one to make it easier to work with
hazard_area_multipolygon_df = gpd.GeoSeries(unary_union(hazrd_area_geoms))
hazard_area_multipolygon = hazard_area_multipolygon_df[0]

In [ ]:
# load networks to visualize
road_network = ox.load_graphml(cache_dir / f"network/drive_graph_{place_name}.graphml")
road_network_removed = ox.load_graphml(cache_dir / f"network/drive_graph_removed{place_name}.graphml")
road_network_remaining = ox.load_graphml(cache_dir / f"network/drive_graph_remaining_{place_name}.graphml")

## Plot the flooded area and the networks

In [ ]:
ax_flooded_boundary = hazard_area_multipolygon_df.plot(figsize=(18, 18), color="#33669999", zorder=10)

In [ ]:
ax_flooded_boundary

In [ ]:
# add liver way in case there is no info about the river bounds
ax_flooded_boundary = waterways.plot(figsize=(18, 18), ax=ax_flooded_boundary, zorder=0)

# add river bounds
ax_flooded_boundary = rivers.plot(figsize=(18, 18), ax=ax_flooded_boundary, zorder=0)

In [ ]:
fig_1, ax_flooded_boundary = ox.plot_graph(
    road_network_removed, figsize=(18, 18), node_size=0, edge_color="red", edge_linewidth=0.25, ax=ax_flooded_boundary
)

fig_1

In [ ]:
# add un_affected_network to the ax
fig_2, ax_flooded_boundary = ox.plot_graph(
    road_network_remaining,
    figsize=(18, 18),
    node_size=0,
    edge_color="black",
    edge_linewidth=0.25,
    show=False,
    close=False,
    ax=ax_flooded_boundary,
)

In [ ]:
fig_2

In [ ]:
# plt.savefig(f"output/flooded_area_trier_multipolygon.png", dpi=150)
fig_2.savefig(output_dir / "plots" / f"affected_network_area_{place_name}.png", dpi=300)

## Create an interactive map with folium containing the flooded areas

### Disclaimer about coordinates:
- Shapely uses: (lon, lat)
- Folium / leaflet uses (lat, lon)

In [ ]:
flooding_map = folium.Map(location=[49.745, 6.65], zoom_start=13, tiles="CartoDB positron")

In [ ]:
# geo_j = polygon1.simplify(tolerance=0.001).to_json()
hazard_area_multipolygon_df.set_crs(epsg=4326, inplace=True)
geo_json_flooded = folium.GeoJson(
    data=hazard_area_multipolygon_df, style_function=lambda x: {"fillColor": "orange", "color": "orange"}
)
folium.Popup("Scenario x years").add_to(geo_json_flooded)

# geo_real = folium.GeoJson(data=boundary,
#                        style_function=lambda x: {'fillColor': 'orange'})
# folium.Popup("Realdata").add_to(geo_real)
geo_json_flooded.add_to(flooding_map)

In [ ]:
flooding_map